# VeggieLens 蔬菜识别模型训练（第二版）

用 **MediaPipe Model Maker** 在两份 Kaggle 数据集（共 25 类）上做迁移学习，产出 App 可直接加载的 `.tflite` 模型。

**这个笔记本会做什么**

1. 合并两份数据集，类名统一成 App 数据库里的英文名（如 `Bitter_Gourd`、`Onion_Red`）
2. 可选：加入“其他”（Other）类，减少对非蔬菜画面的误判
3. 检查训练集与验证/测试集之间的重复图片（避免测试准确率虚高）
4. 训练、导出浮点版和 **int8 量化版**（体积约为浮点版的 1/4）
5. 用导出的 `.tflite` 逐类评估，画混淆矩阵，找出最容易混淆的蔬菜
6. 按“误判率”标定 App 里的三个阈值，直接输出可粘贴的 Kotlin 代码

**运行环境要求**

- Model Maker 依赖 TensorFlow < 2.16，因此需要 **Python 3.9–3.11**
- 建议用 GPU。推荐 **Kaggle Notebook**：两份数据集本来就在 Kaggle 上，可直接“Add Input”挂载，免去下载
- 也可以在 Colab 或本地运行，只需改“步骤 1”的路径

## 步骤 0：环境

In [ ]:
import sys
print(sys.version)
assert sys.version_info[:2] <= (3, 11), (
    "mediapipe-model-maker 依赖 TensorFlow < 2.16，需要 Python 3.9–3.11 的运行环境。"
)

In [ ]:
# 安装依赖。若环境里已预装更新版本的 TensorFlow，这一步会降级它，装完后请重启内核再继续。
!pip install -q mediapipe-model-maker imagehash scikit-learn matplotlib pandas

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "未检测到 GPU（可以训练，但会慢很多）")

## 步骤 1：配置

`DATA_SOURCES` 里每个数据源写：根目录 + 训练 / 验证 / 测试三个子目录名。子目录下应是 **每类一个文件夹**。
路径不确定时，先运行下一格的“查看目录”。

- `OTHER_DIR`：“其他”类图片（水果、手、桌面、包装袋、厨房背景等），**不用分子目录、不用预先划分**，脚本按 70/15/15 自动划分。没有就留 `None`。
- `REAL_PHOTOS_DIR`：自己用手机拍的测试照片，每类一个文件夹（文件夹名用英文类名）。**只用于测试**，最能反映真机效果。没有就留 `None`。

In [ ]:
from pathlib import Path

DATA_SOURCES = [
    {
        "name": "vegetable_image_dataset",  # 15 类
        "root": "/kaggle/input/vegetable-image-dataset/Vegetable Images",
        "splits": {"train": "train", "validation": "validation", "test": "test"},
    },
    {
        "name": "vegetables_nutrionix",  # 10 类
        "root": "/kaggle/input/vegetables-nutrionix",
        "splits": {"train": "train", "validation": "validation", "test": "test"},
    },
]

OTHER_DIR = None        # 例如 "/kaggle/input/veggielens-other"
REAL_PHOTOS_DIR = None  # 例如 "/kaggle/input/veggielens-real-photos"

WORK_DIR = Path("/kaggle/working/veggielens_data")      # 合并后的数据
EXPORT_DIR = Path("/kaggle/working/veggielens_export")  # 模型和报告输出

MODEL_VERSION = "v2"
MODEL_SPEC = "EFFICIENTNET_LITE0"  # 可选 EFFICIENTNET_LITE2（更准、更慢，输入 260×260）
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.0005
FINE_TUNE = True                   # 连同主干网络一起微调，准确率通常明显更高
SEED = 42

# App 中对实拍照片的预处理：取画面中心正方形，边长 = 短边 × 0.8（与 RecognitionConfig.CAMERA_CROP_FRACTION 一致）
CAMERA_CROP_FRACTION = 0.8

# App 数据库中的 25 种蔬菜（模型标签必须与它们一致）
APP_CLASSES = [
    "Bean", "Bitter_Gourd", "Bottle_Gourd", "Brinjal", "Broccoli", "Cabbage", "Capsicum",
    "Carrot", "Cauliflower", "Corn", "Cucumber", "Eggplant", "Galangal", "Garlic", "Ginger",
    "Lettuce", "Onion", "Onion_Red", "Papaya", "Potato", "Pumpkin", "Radish", "Spinach",
    "Sweet_Potato", "Tomato",
]
OTHER_CLASS = "Other"

In [ ]:
# 查看目录：确认 DATA_SOURCES 里的路径和子目录名
for src in DATA_SOURCES:
    root = Path(src["root"])
    print(f"== {src['name']}: {root}  存在={root.exists()}")
    if root.exists():
        for child in sorted(root.iterdir())[:10]:
            print("   ", child.name)

## 步骤 2：合并数据集

类名统一规则：按空格、下划线、连字符拆词，每个词首字母大写，再用下划线连接。
例如 `Bitter Gourd` → `Bitter_Gourd`，`Sweet potato` → `Sweet_Potato`，`Onion Red` → `Onion_Red`。

In [ ]:
import collections
import hashlib
import re
import shutil

import pandas as pd

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def canonical(name: str) -> str:
    words = [w for w in re.split(r"[\s_\-]+", name.strip()) if w]
    return "_".join(w.capitalize() for w in words)


CANON_TO_APP = {canonical(c): c for c in APP_CLASSES}
CANON_TO_APP[canonical(OTHER_CLASS)] = OTHER_CLASS


def images_in(folder: Path):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)


if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

counts = collections.defaultdict(collections.Counter)
unknown_folders = collections.Counter()

for src in DATA_SOURCES:
    for split, sub in src["splits"].items():
        split_dir = Path(src["root"]) / sub
        assert split_dir.is_dir(), f"找不到 {split_dir}，请检查 DATA_SOURCES"
        for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
            label = CANON_TO_APP.get(canonical(class_dir.name))
            if label is None:
                unknown_folders[f"{src['name']}/{sub}/{class_dir.name}"] += 1
                continue
            out = WORK_DIR / split / label
            out.mkdir(parents=True, exist_ok=True)
            for img in images_in(class_dir):
                shutil.copy2(img, out / f"{src['name']}__{img.name}")
                counts[split][label] += 1

if OTHER_DIR:
    # 按文件名哈希确定性地划分 70/15/15，重复运行结果一致
    for img in sorted(p for p in Path(OTHER_DIR).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT):
        bucket = int(hashlib.md5(img.name.encode()).hexdigest(), 16) % 100
        split = "train" if bucket < 70 else ("validation" if bucket < 85 else "test")
        out = WORK_DIR / split / OTHER_CLASS
        out.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, out / f"other__{img.name}")
        counts[split][OTHER_CLASS] += 1

if unknown_folders:
    print("⚠️ 以下文件夹不在 APP_CLASSES 中，已跳过：")
    for k in unknown_folders:
        print("   ", k)

table = pd.DataFrame(counts).fillna(0).astype(int).sort_index()
table["total"] = table.sum(axis=1)
display(table)

expected = set(APP_CLASSES) | ({OTHER_CLASS} if OTHER_DIR else set())
for split in ["train", "validation", "test"]:
    missing = expected - set(counts[split])
    assert not missing, f"{split} 缺少类别：{sorted(missing)}"
print("✅ 三个划分都包含全部类别")

## 步骤 3（建议）：检查重复图片

公开数据集里常有同一张图出现在训练集和测试集，会让测试准确率虚高。这里用感知哈希找出 **与训练集完全相同** 的验证/测试图片，并把它们移除。

In [ ]:
import imagehash
from PIL import Image

REMOVE_DUPLICATES = True


def phash(path: Path) -> str:
    with Image.open(path) as im:
        return str(imagehash.phash(im.convert("RGB")))


train_hashes = {phash(p) for p in (WORK_DIR / "train").rglob("*") if p.is_file()}
duplicates = [
    (split, p)
    for split in ["validation", "test"]
    for p in (WORK_DIR / split).rglob("*")
    if p.is_file() and phash(p) in train_hashes
]
print(f"与训练集重复的图片：{len(duplicates)} 张")
for split, n in collections.Counter(s for s, _ in duplicates).items():
    print(f"   {split}: {n}")

if REMOVE_DUPLICATES:
    for _, p in duplicates:
        p.unlink()
    print("已移除。")

## 步骤 4：训练

In [ ]:
from mediapipe_model_maker import image_classifier, quantization

train_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "train"))
validation_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "validation"))
test_data = image_classifier.Dataset.from_folder(str(WORK_DIR / "test"), shuffle=False)

assert train_data.label_names == validation_data.label_names == test_data.label_names
print("类别数：", train_data.num_classes)
print("标签：", train_data.label_names)
print("训练 / 验证 / 测试：", len(train_data), len(validation_data), len(test_data))

In [ ]:
# Model Maker 会从 export_dir/checkpoint 续训，换参数重训前先清空，避免加载旧权重
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir(parents=True)

tf.keras.utils.set_random_seed(SEED)

hparams = image_classifier.HParams(
    export_dir=str(EXPORT_DIR),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    do_fine_tuning=FINE_TUNE,
    do_data_augmentation=True,  # 随机裁剪、翻转等
    label_smoothing=0.1,
    shuffle=True,
)
options = image_classifier.ImageClassifierOptions(
    supported_model=getattr(image_classifier.SupportedModels, MODEL_SPEC),
    hparams=hparams,
    model_options=image_classifier.ModelOptions(dropout_rate=0.2),
)

model = image_classifier.ImageClassifier.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options,
)

In [ ]:
metrics = model.evaluate(test_data, batch_size=BATCH_SIZE)
print("测试集（Keras 模型）：loss = %.4f, accuracy = %.4f" % (metrics[0], metrics[1]))

## 步骤 5：导出模型

In [ ]:
FLOAT_NAME = f"veggielens_classifier_{MODEL_VERSION}_float.tflite"
INT8_NAME = f"veggielens_classifier_{MODEL_VERSION}.tflite"

model.export_model(model_name=FLOAT_NAME)

int8_config = quantization.QuantizationConfig.for_int8(representative_data=validation_data)
model.export_model(model_name=INT8_NAME, quantization_config=int8_config)

for name in [FLOAT_NAME, INT8_NAME]:
    path = EXPORT_DIR / name
    print(f"{name}: {path.stat().st_size / 1024 / 1024:.1f} MB")

## 步骤 6：用导出的 .tflite 逐类评估

这里用 App 里同一个推理库（MediaPipe Tasks）加载导出的模型，结果就是 App 实际会得到的分数。
同时评估浮点版和 int8 版，确认量化没有明显掉点。

In [ ]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision


def load_classifier(model_path: Path):
    opts = vision.ImageClassifierOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=str(model_path)),
        max_results=3,
    )
    return vision.ImageClassifier.create_from_options(opts)


def center_crop(im: Image.Image, fraction: float) -> Image.Image:
    side = int(min(im.size) * fraction)
    left = (im.width - side) // 2
    top = (im.height - side) // 2
    return im.crop((left, top, left + side, top + side))


def predict_folder(classifier, root: Path, crop_fraction=None) -> pd.DataFrame:
    rows = []
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        true_label = CANON_TO_APP.get(canonical(class_dir.name))
        if true_label is None:
            print("跳过未知类别文件夹：", class_dir.name)
            continue
        for path in images_in(class_dir):
            with Image.open(path) as im:
                im = im.convert("RGB")
                if crop_fraction:
                    im = center_crop(im, crop_fraction)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.asarray(im))
            categories = classifier.classify(mp_image).classifications[0].categories
            top = categories[0] if categories else None
            rows.append({
                "file": str(path),
                "true": true_label,
                "pred": top.category_name if top else None,
                "score": top.score if top else 0.0,
                "top3": [c.category_name for c in categories],
            })
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame, title: str):
    top1 = (df["pred"] == df["true"]).mean()
    top3 = np.mean([t in p for t, p in zip(df["true"], df["top3"])])
    print(f"{title}：{len(df)} 张，Top-1 准确率 {top1:.2%}，Top-3 准确率 {top3:.2%}")
    return top1


results = {}
for name in [FLOAT_NAME, INT8_NAME]:
    clf = load_classifier(EXPORT_DIR / name)
    results[name] = predict_folder(clf, WORK_DIR / "test")
    summarize(results[name], f"测试集 · {name}")

if REAL_PHOTOS_DIR:
    clf = load_classifier(EXPORT_DIR / INT8_NAME)
    results["real"] = predict_folder(clf, Path(REAL_PHOTOS_DIR), crop_fraction=CAMERA_CROP_FRACTION)
    summarize(results["real"], f"手机实拍 · {INT8_NAME}（已按 App 的方式中心裁剪）")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

EVAL_KEY = "real" if "real" in results else INT8_NAME
df = results[EVAL_KEY]
labels = sorted(set(df["true"]) | set(df["pred"].dropna()))

print(f"逐类结果（{EVAL_KEY}）")
print(classification_report(df["true"], df["pred"].fillna("None"), labels=labels, digits=3, zero_division=0))

cm = confusion_matrix(df["true"], df["pred"].fillna("None"), labels=labels)
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm / np.maximum(cm.sum(axis=1, keepdims=True), 1), cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=90)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix (row-normalized)")
fig.colorbar(im)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

wrong = df[df["pred"] != df["true"]]
print("最常见的误判（真实 → 预测）：")
display(wrong.groupby(["true", "pred"]).size().sort_values(ascending=False).head(15).rename("次数"))

## 步骤 7：标定 App 的阈值

App 里有三个阈值（单位是“匹配度百分比”= 分数 × 100）：

| 阈值 | 用途 | 标定目标 |
| --- | --- | --- |
| `AUTO_SCAN_MIN_CONFIDENCE` | 自动识别直接出结果 | 出结果时准确率 ≥ 97% |
| `MANUAL_SCAN_MIN_CONFIDENCE` | 手动拍照 / 相册识别 | 准确率 ≥ 90% |
| `HINT_MIN_CONFIDENCE` | 取景时提示“像是 xx” | 准确率 ≥ 75% |

“覆盖率”是蔬菜图片中能拿到结果的比例。阈值越高越准，但覆盖率越低。有手机实拍集时优先用它标定。

In [ ]:
import json

def calibration_table(df: pd.DataFrame) -> pd.DataFrame:
    veg = df[df["true"] != OTHER_CLASS]
    rows = []
    for t in range(5, 100, 5):
        accepted = df[(df["pred"] != OTHER_CLASS) & (df["score"] * 100 >= t)]
        accepted_veg = accepted[accepted["true"] != OTHER_CLASS]
        rows.append({
            "阈值": t,
            "准确率": (accepted["pred"] == accepted["true"]).mean() if len(accepted) else np.nan,
            "覆盖率": len(accepted_veg) / max(len(veg), 1),
        })
    return pd.DataFrame(rows)


def pick_threshold(table: pd.DataFrame, target: float, fallback: int) -> int:
    ok = table[table["准确率"] >= target]
    return int(ok["阈值"].min()) if len(ok) else fallback


cal = calibration_table(results[EVAL_KEY])
display(cal.round(3))

thresholds = {
    "AUTO_SCAN_MIN_CONFIDENCE": pick_threshold(cal, 0.97, 85),
    "MANUAL_SCAN_MIN_CONFIDENCE": pick_threshold(cal, 0.90, 50),
    "HINT_MIN_CONFIDENCE": pick_threshold(cal, 0.75, 30),
    "ALTERNATIVE_MIN_CONFIDENCE": 15,
}

print("\n把下面几行替换到 app/src/main/java/com/example/veggielens/viewmodel/RecognitionConfig.kt：\n")
print(f'    const val MODEL_ASSET = "{INT8_NAME}"')
for key, value in thresholds.items():
    print(f"    const val {key} = {value}")

report = {
    "model": INT8_NAME,
    "model_spec": MODEL_SPEC,
    "epochs": EPOCHS,
    "labels": train_data.label_names,
    "class_counts": {split: dict(c) for split, c in counts.items()},
    "test_top1_float": float((results[FLOAT_NAME]["pred"] == results[FLOAT_NAME]["true"]).mean()),
    "test_top1_int8": float((results[INT8_NAME]["pred"] == results[INT8_NAME]["true"]).mean()),
    "real_top1": float((results["real"]["pred"] == results["real"]["true"]).mean()) if "real" in results else None,
    "thresholds": thresholds,
}
(EXPORT_DIR / "training_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
print("\n报告已保存：", EXPORT_DIR / "training_report.json")

## 步骤 8：放进 App

1. 下载 `veggielens_export/` 里的 `veggielens_classifier_v2.tflite`（int8 版），放到 `app/src/main/assets/`
2. 按上一步输出更新 `RecognitionConfig.kt` 里的 `MODEL_ASSET` 和阈值
3. 运行单元测试：`./gradlew test`（`VegetableMatcherTest` 会检查每个标签都能对应到正确的蔬菜）
4. 真机上对着几种蔬菜试一试，尤其是混淆矩阵里最容易混的几对
5. 确认没问题后，可以删除旧的 `veggielens_classifier_v1.tflite` 以减小安装包

`confusion_matrix.png` 和 `training_report.json` 建议一起保存，方便和下一版对比。